# Prerequisite 06 · PyTorch：从 tensor 到完整训练循环

[上一章：机器学习](Prerequisite05_Machine_Learning.ipynb) · [下一章：Gymnasium](Prerequisite07_Gymnasium.ipynb)

**目标：**理解 tensor、autograd、`nn.Module`、optimizer、`detach/no_grad`、device 和概率分布；独立写出一个小型训练循环。


## 1. Tensor 与 NumPy 的联系

Tensor 同样有 shape、dtype 和广播，还能记录计算图并在 CPU/GPU 上计算。RL 中通常把 NumPy observation 转成 float tensor，再送入网络。


In [1]:
import torch

torch.manual_seed(6)
states = torch.tensor([[.1, .2, -.3], [0., .4, .5]], dtype=torch.float32)
linear = torch.nn.Linear(3, 2)
q_values = linear(states)
actions = torch.tensor([1, 0])
chosen_q = q_values.gather(1, actions.unsqueeze(1)).squeeze(1)
print("Q shape:", tuple(q_values.shape), "chosen Q shape:", tuple(chosen_q.shape))


Q shape: (2, 2) chosen Q shape: (2,)


## 2. Autograd：计算图上的链式法则

`loss.backward()` 把梯度累积到参数的 `.grad`；`optimizer.step()` 才真正修改参数。默认梯度会累加，所以每轮先 `zero_grad()`。


In [2]:
w = torch.tensor(1.5, requires_grad=True)
x, y = torch.tensor(3.0), torch.tensor(7.0)
loss = (w*x - y)**2
loss.backward()
print("loss:", loss.item(), "dL/dw:", w.grad.item())
assert abs(w.grad.item() - 2*(1.5*3-7)*3) < 1e-6


loss: 6.25 dL/dw: -15.0


## 3. 完整训练循环

顺序固定为：准备 batch → forward → loss → `zero_grad` → `backward` → `step`。评估时使用 `model.eval()` 与 `torch.no_grad()`。


In [3]:
x = torch.linspace(-1, 1, 100).unsqueeze(1)
y = 2.5*x - .4 + .08*torch.randn_like(x)
model = torch.nn.Sequential(torch.nn.Linear(1, 16), torch.nn.Tanh(), torch.nn.Linear(16, 1))
optimizer = torch.optim.Adam(model.parameters(), lr=.03)
history = []
for _ in range(120):
    prediction = model(x)
    loss = torch.nn.functional.mse_loss(prediction, y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    history.append(loss.item())

model.eval()
with torch.no_grad():
    final_mse = torch.nn.functional.mse_loss(model(x), y).item()
print("first/final loss:", round(history[0], 4), round(final_mse, 4))
assert final_mse < history[0] * .1


first/final loss: 2.8891 0.0088


## 4. detach、no_grad 与概率分布

- `detach()` 返回共享数据但不继续追踪当前计算图，常用于 TD target 或 advantage target。
- `no_grad()` 在整个代码块中关闭梯度，适合 target network 和评估。
- `torch.distributions.Categorical/Normal` 提供 `sample`、`log_prob`、`entropy`，是策略梯度的基础。


In [4]:
logits = torch.tensor([.2, 1.0, -.4], requires_grad=True)
distribution = torch.distributions.Categorical(logits=logits)
action = distribution.sample()
objective = -distribution.log_prob(action) * torch.tensor(1.5)
objective.backward()
print("sample/log_prob/entropy:", action.item(), distribution.log_prob(action).item(), distribution.entropy().item())
print("logit gradient:", logits.grad)


sample/log_prob/entropy: 2 -1.928228735923767 0.9437533020973206
logit gradient: tensor([ 0.3974,  0.8845, -1.2819])


## 初学者常见坑与过关标准

- 把 batch 维 `squeeze` 掉，单样本能跑、多样本失败。
- 忘记 `zero_grad()`；对 target 路径忘记 `detach/no_grad()`。
- NumPy 默认 float64，网络参数通常 float32，dtype 不一致。
- 保存整个临时计算图导致内存持续增长。

**练习：**把回归模型改成一层线性网络；比较 SGD 和 Adam，但保持数据、初始化、步数一致。

**过关：**能不看模板写出完整训练循环；能解释何时使用 `detach`；会检查 tensor shape、dtype、device 和 gradient。

**官方选读：**[PyTorch Learn the Basics](https://docs.pytorch.org/tutorials/beginner/basics/intro.html)，依次看 Tensors、Build Model、Autograd、Optimization、Save/Load。
